# TensorFlow Tutorial

В предыдущих лабораторных работах вы всегда использовали numpy для построения нейронных сетей.
Теперь мы изучим систему/фреймворк глубокого обучения, который позволит легче строить нейронные сети.
Фреймворки машинного обучения, такие как TensorFlow, Torch, Caffe, Keras и многие другие, позволяют значительно ускорить развитие машинного обучения.
Все эти фреймворки также имеют много документации, которую вы должны свободно читать.
В этом задании вы научитесь делать следующее в TensorFlow:

**В рамках данной лабораторной работы будут приобретены следующие навыки (знания):**
- Использование `tf.Variable` для модификации состояния переменной
- Обучите нейронную сеть с использованием TensorFlow

`Данный материал опирается и использует материалы курса Deep Learning от организации deeplearning.ai`

 Ссылка на основной курс (для желающих получить дополнительный сертификаты): https://www.coursera.org/specializations/deep-learning


## 1 -  Пакеты/Библиотеки

In [ ]:
import h5py
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.python.framework.ops import EagerTensor
from tensorflow.python.ops.resource_variable_ops import ResourceVariable
import time
%matplotlib inline


### 1.1 - Проверим версию TensorFlow 

In [ ]:
tf.__version__

## 2 - Оптимизация с использованием GradientTape

Красота TensorFlow v2+ заключается в его простоте. По сути, все, что вам нужно сделать, это реализовать прямое распространение через вычислительный график. TensorFlow вычислит производные для вас, двигаясь назад по графику, записанному с помощью `GradientTape`. Все, что вам остается сделать, это указать функцию затрат и оптимизатор, которые вы хотите использовать.

При написании программы с использованием библиотеки TensorFlow основным объектом для использования и преобразования является `tf.Тензор`. Эти тензоры являются эквивалентом тензорного потока массивов Numpy, т.е. многомерных массивов заданного типа данных, которые также содержат информацию о вычислительном графике.

Ниже вы будете использовать `tf.Variable` для хранения состояния ваших переменных. Переменные могут быть созданы только один раз, так как их начальное значение определяет форму и тип переменной. Кроме того, аргумент `dtype` в `tf.Variable` может быть установлен, чтобы разрешить преобразование данных в этот тип.

Здесь вы вызовете набор данных TensorFlow, созданный в файле HDF5, который вы можете использовать вместо массива Numpy для хранения ваших наборов данных.

Вы будете использовать набор данных о знаках рук, состоящий из изображений с формой 64x64x3.

In [ ]:
train_dataset = h5py.File('datasets/train_signs.h5', "r")
test_dataset = h5py.File('datasets/test_signs.h5', "r")

In [ ]:
x_train = tf.data.Dataset.from_tensor_slices(train_dataset['train_set_x'])
y_train = tf.data.Dataset.from_tensor_slices(train_dataset['train_set_y'])

x_test = tf.data.Dataset.from_tensor_slices(test_dataset['test_set_x'])
y_test = tf.data.Dataset.from_tensor_slices(test_dataset['test_set_y'])

In [ ]:
type(x_train)

Поскольку наборы данных TensorFlow являются генераторами, вы не можете получить прямой доступ к содержимому, если не выполните итерацию по ним в цикле for или явно создав итератор Python с использованием `iter` и потребляя его элементы с помощью `next`. Кроме того, вы можете проверить `shape` и `dtype` каждого элемента, используя атрибут `element_spec`.

In [ ]:
print(x_train.element_spec)

In [ ]:
print(next(iter(x_train)))

Набор данных, который вы будете использовать во время этого задания, представляет собой подмножество цифр языка жестов. Он содержит шесть различных классов, представляющих цифры от 0 до 5.

In [ ]:
unique_labels = set()
for element in y_train:
    unique_labels.add(element.numpy())
print(unique_labels)

In [ ]:
images_iter = iter(x_train)
labels_iter = iter(y_train)
plt.figure(figsize=(10, 10))
for i in range(25):
    ax = plt.subplot(5, 5, i + 1)
    plt.imshow(next(images_iter).numpy().astype("uint8"))
    plt.title(next(labels_iter).numpy().astype("uint8"))
    plt.axis("off")

Есть еще одно дополнительное различие между наборами данных TensorFlow и массивами Numpy: если вам нужно их преобразовать, вы должны вызвать метод `map`, чтобы применить функцию, переданную в качестве аргумента, к каждому из элементов.

In [ ]:
def normalize(image):
    """
    Преобразование изображения в размер (64 * 64 * 3, )
    и нормализация компонентов.
    
    Arguments
    image - входной тензор.
    
    Returns: 
    result -- преобразованный тензор
    """
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.reshape(image, [-1,])
    return image

In [ ]:
new_train = x_train.map(normalize)
new_test = x_test.map(normalize)

In [ ]:
new_train.element_spec

In [ ]:
print(next(iter(new_train)))

### 2.1 - Линейная функция

Давайте начнем это упражнение с вычисления следующего уравнения: $Y = WX + b$, где $W$ и $X$ случайные матрицы и b вектор случайных значений. 

**Упражнение** Вычислить $WX + b$ где $W, X$, и $b$ получаются из случайного нормального распределения. W размером (4, 3), X - (3,1) и b - (4,1). В качестве примера, вот как можно определить константу X, которая имеет форму (3,1):
```python
X = tf.constant(np.random.randn(3,1), name = "X")
```
Обратите внимание, что разница между `tf.constant` и `tf.Variable` заключается в том, что вы можете изменить состояние `tf.Variable` но не можете изменить состояние `tf.constant`.

Возможно, вам помогут следующие функции: 
- tf.matmul(..., ...) перемножение матриц
- tf.add(..., ...) сложение
- np.random.randn(...) случайная инициализация

In [ ]:
# ОЦЕНИВАЕМОЕ: linear_function

def linear_function():
    """
    Реализация линейной функции: 
            инициализация W, который должен быть случайным тензором размером (4,3)
            инициализация X, который должен быть случайным тензором размером (3,1)
            инициализация b, который должен быть случайным тензором размером (4,1)
    Returns: 
    result -- Y = WX + b 
    """
    
    np.random.seed(1)
    
    ### НАЧАЛО ВАШЕГО КОД ЗДЕСЬ ### (4 строки кода)
     # Инициализация случайных тензоров
    X = tf.constant(np.random.randn(3, 1), name="X", dtype=tf.float32)
    W = tf.constant(np.random.randn(4, 3), name="W", dtype=tf.float32)
    b = tf.constant(np.random.randn(4, 1), name="b", dtype=tf.float32)

    # Вычисление Y
    Y = tf.add(tf.matmul(W, X), b)
    ### ОКОНЧАНИЕ ВАШЕГО КОД ЗДЕСЬ ### 
    
    return Y

In [ ]:
result = linear_function()
print(result)

assert type(result) == EagerTensor, "Требуется использовать TensorFlow API"
assert np.allclose(result, [[-2.15657382], [ 2.95891446], [-1.08926781], [-0.84538042]]), "Error"

#### ***Ожидаемый результат***: 

```
result = 
[[-2.15657382]
 [ 2.95891446]
 [-1.08926781]
 [-0.84538042]]
```

### 2.2 - Вычисление sigmoid 

Ранее вы реализовали линейную функцию. Tensorflow предлагает множество часто используемых функций нейронной сети, таких как `tf.sigmoid` и `tf.softmax`. Для этого упражнения вычислим сигмовидную функцию входного сигнала.

**Упражнение** В этом упражнении требуется привести свой тензор к типу `float32`, используя `tf.cast`, затем вычислите сигмоиду, используя `tf.keras.activations.sigmoid`.

Вы должны использовать следующее:
- `tf.cast("...", tf.float32)`
- `tf.keras.activations.sigmoid("...")`

In [ ]:
# ОЦЕНИВАЕМОЕ: sigmoid

def sigmoid(z): 
    """
    Вычисление sigmoid для z
    
    Arguments:
    z -- входное значение, скаляр или вектор
    
    Returns: 
    results -- sigmoid по z
    """
    # tf.keras.activations.sigmoid принимает на вход float16, float32, float64, complex64, or complex128.
    # НАЧАЛО ВАШЕГО КОД ЗДЕСЬ
    z = tf.cast(z, tf.float32)  # Приводим z к типу float32
    a = tf.keras.activations.sigmoid(z)  # Вычисляем сигмоиду
    # ОКОНЧАНИЕ ВАШЕГО КОД ЗДЕСЬ
    return a

In [ ]:
result = sigmoid(-1)
print ("type: " + str(type(result)))
print ("dtype: " + str(result.dtype))
print ("sigmoid(-1) = " + str(result))
print ("sigmoid(0) = " + str(sigmoid(0.0)))
print ("sigmoid(12) = " + str(sigmoid(12)))

def sigmoid_test(target):
    result = target(0)
    assert(type(result) == EagerTensor)
    assert (result.dtype == tf.float32)
    assert sigmoid(0) == 0.5, "Error"
    assert sigmoid(-1) == 0.26894143, "Error"
    assert sigmoid(12) == 0.99999386, "Error"

sigmoid_test(sigmoid)

**Ожидаемый результат**: 
<table>
<tr> 
<td>
type
</td>
<td>
class 'tensorflow.python.framework.ops.EagerTensor'
</td>
</tr><tr> 
<td>
dtype
</td>
<td>
"dtype: 'float32'
</td>
</tr>
<tr> 
<td>
Sigmoid(-1)
</td>
<td>
0.2689414
</td>
</tr>
<tr> 
<td>
Sigmoid(0)
</td>
<td>
0.5
</td>
</tr>
<tr> 
<td>
Sigmoid(12)
</td>
<td>
0.9999938
</td>
</tr> 

</table> 

### 2.3 - Реализация One Hot encodings

Много раз в глубоком обучении вы будете иметь вектор y с числами в диапазоне от 0 до C-1, где C-количество классов. Если C, например, 4, то у вас может быть следующий вектор y, который вам нужно будет преобразовать следующим образом:

<img src="images/onehot.png" style="width:600px;height:150px;">

Это называется "one hot" кодировка, потому что в преобразованном представлении ровно один элемент каждого столбца является "hot" (то есть равным 1). Чтобы сделать это преобразование в numpy, вам, возможно, придется написать несколько строки кода. В tensorflow можно использовать одну строку кода:

- [tf.one_hot(labels, depth, axis=0)](https://www.tensorflow.org/api_docs/python/tf/one_hot)

**Упражнение** Реализуйте функцию ниже, чтобы взять один вектор меток и общее число классов $C$, и вернуть one hot encoding. Используйте `tf.one_hot()` для этого и `tf.reshape()`, чтобы изменить форму выходного результата.
- `tf.reshape(tensor, shape)`

In [ ]:
# ОЦЕНИВАЕМОЕ: one_hot_matrix
def one_hot_matrix(label, depth=6):
    """
    Вычисление one hot encoding
    
    Arguments:
        label --  (int) категориальные метки
        depth --  (int) количество различных классов
    
    Returns:
         one_hot -- tf.Tensor матрица one-hot encoding.
    """
    # НАЧАЛО ВАШЕГО КОД ЗДЕСЬ
    one_hot = tf.one_hot(label, depth)
    one_hot = tf.reshape(one_hot, (depth,)) 
    # ОКОНЧАНИЕ ВАШЕГО КОД ЗДЕСЬ
    return one_hot

In [ ]:
def one_hot_matrix_test(target):
    label = tf.constant(1)
    depth = 4
    result = target(label, depth)
    print("Тест 1:",result)
    assert result.shape[0] == depth, "Требуется использовать параметр глубины"
    assert np.allclose(result, [0., 1. ,0., 0.] ), "Неправильный ответ. Использовать tf.one_hot"
    label_2 = [2]
    result = target(label_2, depth)
    print("Test 2:", result)
    assert result.shape[0] == depth, "Требуется использовать параметр глубины"
    assert np.allclose(result, [0., 0. ,1., 0.] ), "Неправильный ответ. Используйте tf.reshape as instructed"
    

one_hot_matrix_test(one_hot_matrix)

**Ожидаемый результат**
```
Тест 1: tf.Tensor([0. 1. 0. 0.], shape=(4,), dtype=float32)
Тест 2: tf.Tensor([0. 0. 1. 0.], shape=(4,), dtype=float32)
```

In [ ]:
new_y_test = y_test.map(one_hot_matrix)
new_y_train = y_train.map(one_hot_matrix)

In [ ]:
print(next(iter(new_y_test)))

### 2.4 - Инициализация нулями и единицами

Теперь вы инициализируете вектор чисел с помощью инициализатора Glorot. Функция, которую вы будете вызывать, называется `tf.keras.initializers.GlorotNormal`, который извлекает выборки из усеченного нормального распределения с центром в 0, с `stddev = sqrt(2 / (fan_in + fan_out))`, где `fan_in` количество входных единиц, а `fan_out` количество выходных единиц. 

Для инициализации нулями или единицами вы можете использовать `tf.zeros()` или `tf.ones()`. 

**Упражнение** Реализуйте приведенную ниже функцию, чтобы принять форму и вернуть массив чисел, используя инициализатор GlorotNormal.
 - `tf.keras.initializers.GlorotNormal(seed=1)`
 - `tf.Variable(initializer(shape=())`

In [ ]:
# ОЦЕНИВАЕМОЕ: initialize_parameters

def initialize_parameters():
    """
    Инициализирует параметры для построения нейронной сети с помощью TensorFlow. Формы являются:
                        W1 : [25, 12288]
                        b1 : [25, 1]
                        W2 : [12, 25]
                        b2 : [12, 1]
                        W3 : [6, 12]
                        b3 : [6, 1]
    
    Returns:
    parameters -- словарь, содержащий W1, b1, W2, b2, W3, b3
    """
    initializer = tf.keras.initializers.GlorotNormal(seed=23)   
    # НАЧАЛО ВАШЕГО КОД ЗДЕСЬ
    W1 = tf.Variable(initializer(shape=(25, 12288)))
    b1 = tf.Variable(initializer(shape=(25, 1)))
    W2 = tf.Variable(initializer(shape=(12, 25)))
    b2 = tf.Variable(initializer(shape=(12, 1)))
    W3 = tf.Variable(initializer(shape=(6, 12)))
    b3 = tf.Variable(initializer(shape=(6, 1)))
    # ОКОНЧАНИЕ ВАШЕГО КОД ЗДЕСЬ

    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2,
                  "W3": W3,
                  "b3": b3}
    
    return parameters

In [ ]:
def initialize_parameters_test(target):
    parameters = target()

    values = {"W1": (25, 12288),
              "b1": (25, 1),
              "W2": (12, 25),
              "b2": (12, 1),
              "W3": (6, 12),
              "b3": (6, 1)}
    for key in parameters:
        print(f"{key} shape: {tuple(parameters[key].shape)}")
        assert type(parameters[key]) == ResourceVariable, "All parameter must be created using tf.Variable"
        assert tuple(parameters[key].shape) == values[key], f"{key}: wrong shape"
        assert np.abs(np.mean(parameters[key].numpy())) < 0.5,  f"{key}: Use the GlorotNormal initializer"
        assert np.std(parameters[key].numpy()) > 0 and np.std(parameters[key].numpy()) < 1, f"{key}: Use the GlorotNormal initializer"

    
initialize_parameters_test(initialize_parameters)

**Ожидаемый результат**: 
```
W1 shape: (25, 12288)
b1 shape: (25, 1)
W2 shape: (12, 25)
b2 shape: (12, 1)
W3 shape: (6, 12)
b3 shape: (6, 1)
```

In [ ]:
parameters = initialize_parameters()

# 3 - Построение нейронной сети с использованием tensorflow

В данной части лабораторной работы необходимо построить нейронную сеть с использованием tensorflow. Помните, что существует две части для реализации модели:

- Реализация прямого распространения
- Извлеките градиенты и обучите модель

### 3.1 - Реализация прямого распространения

Одна из сильных сторон TensorFlow заключается в том, что нужно только реализовать функцию прямого распространения, и она будет автоматически отслеживать операции, которые вы выполнили для вычисления обратного распространения.


**Упражнение** Реализуйте функцию `forward_propagation`.

**Замечание** Использовать можно только TF API. 

- tf.math.add
- tf.linalg.matmul
- tf.keras.activations.relu


In [ ]:
# ОЦЕНИВАЕМОЕ: forward_propagation

def forward_propagation(X, parameters):
    """
    Реализация прямого распространения по нейронной сети с архитектурой: LINEAR -> RELU -> LINEAR -> RELU -> LINEAR
    
    Arguments:
    X -- заполнитель входного набора данных, формы (размер ввода, количество примеров)
    parameters -- словарь с параметрами "W1", "b1", "W2", "b2", "W3", "b3"

    Returns:
    Z3 -- выход с последней LINEAR нейрона
    """
    # НАЧАЛО ВАШЕГО КОД ЗДЕСЬ
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]
    
                       # Numpy эквивалент:
    Z1 = tf.math.add(tf.linalg.matmul(W1, X), b1)          # Z1 = np.dot(W1, X) + b1
    A1 = tf.keras.activations.relu(Z1)         # A1 = relu(Z1)
    Z2 = tf.math.add(tf.linalg.matmul(W2, A1), b2)          # Z2 = np.dot(W2, A1) + b2
    A2 = tf.keras.activations.relu(Z2)          # A2 = relu(Z2)
    Z3 = tf.math.add(tf.linalg.matmul(W3, A2), b3)         # Z3 = np.dot(W3, A2) + b3
    # ОКОНЧАНИЕ ВАШЕГО КОД ЗДЕСЬ
    
    return Z3

In [ ]:
def forward_propagation_test(target, examples):
    minibatches = examples.batch(2)
    for minibatch in minibatches:
        forward_pass = target(tf.transpose(minibatch), parameters)
        print(forward_pass)
        assert type(forward_pass) == EagerTensor, "Your output is not a tensor"
        assert forward_pass.shape == (6, 2), "Last layer must use W3 and b3"
        assert np.allclose(forward_pass, 
                           [[ 0.63760024,  0.65318763],
                            [-0.40075475, -0.36886692],
                            [-0.2898944,  -0.33599067],
                            [ 0.7001549,   0.82784766],
                            [ 0.86038524,  0.97037446],
                            [ 0.8381866,   0.77697194]]), "Output does not match"
        break
    
forward_propagation_test(forward_propagation, new_train)

**Ожидаемый результат**: 
```
tf.Tensor(
[[ 0.63760024,  0.65318763],
[-0.40075475, -0.36886692],
[-0.2898944,  -0.33599067],
[ 0.7001549,   0.82784766],
[ 0.86038524,  0.97037446],
[ 0.8381866,   0.77697194]], shape=(6, 2), dtype=float32)
```

### 3.2 Вычисление функции потерь

Все, что вам нужно сделать сейчас, это определить функцию потерь, которую вы собираетесь использовать. В этом случае, поскольку у нас есть проблема классификации с 6 метками, будет работать категориальная перекрестная энтропия.

**Упражнение** Реализуйте функцию потерь
- Важное замечание `y_pred` и `y_true` являются входными параметрами [tf.keras.losses.categorical_crossentropy](https://www.tensorflow.org/api_docs/python/tf/keras/losses/categorical_crossentropy) aожидается, что они будут иметь форму (количество примеров, количество классов). 

- `tf.reduce_mean` в основном выполняется суммирование по примерам.

In [ ]:
# ОЦЕНИВАЕМОЕ: compute_cost 

def compute_cost(logits, labels):
    """
    Вычисление значения функции потерь
    
    Arguments:
    logits -- выход из функции forward_propagation, с размером (6, кол-во примеров)
    labels -- "true" вектор меток
    
    Returns:
    cost - Tensor of the cost function
    """
    
    # НАЧАЛО ВАШЕГО КОД ЗДЕСЬ
    cost = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(labels, logits, from_logits=True))   
    # ОКОНЧАНИЕ ВАШЕГО КОД ЗДЕСЬ
    return cost

In [ ]:
def compute_cost_test(target, Y):
    pred = tf.constant([[ 2.4048107,   5.0334096 ],
             [-0.7921977,  -4.1523376 ],
             [ 0.9447198,  -0.46802214],
             [ 1.158121,    3.9810789 ],
             [ 4.768706,    2.3220146 ],
             [ 6.1481323,   3.909829  ]])
    minibatches = Y.batch(2)
    for minibatch in minibatches:
        result = target(pred, tf.transpose(minibatch))
        break
        
    print(result)
    assert(type(result) == EagerTensor), "Требуется использовать TF API"
    assert (np.abs(result - (0.25361037 + 0.5566767) / 2.0) < 1e-6), "Test does not match. Did you get the mean of your cost functions?"


compute_cost_test(compute_cost, new_y_train)

**Ожидаемый результат**: 

```
tf.Tensor(0.40514335, shape=(), dtype=float32)
```

### 3.3 - Обучение модели

Для определения способа оптимизации используйте функцию - `tf.keras.optimizers.Adam`, а затем вызвать его в цикле обучения. 

Обратите внимание, что функция `tape.gradient` позволяет вам извлекать операции, записанные для автоматической дифференциации внутри блока `GradientTape`. Затем, вызвав метод оптимизатора `apply_gradients`, вы примените правила обновления оптимизатора к каждому обучаемому параметру.


Здесь вы должны принять к сведению важный дополнительный шаг, который был добавлен в процесс пакетного обучения:
- `tf.Data.dataset = dataset.prefetch(8)` 


Это предотвращает узкое место в памяти, которое может возникнуть при чтении с диска. `prefetch()` достает данные, когда это необходимо.

In [ ]:
def model(X_train, Y_train, X_test, Y_test, learning_rate = 0.0001,
          num_epochs = 1500, minibatch_size = 32, print_cost = True):
    """
    Реализация 3-слойнной нейронной сети: LINEAR->RELU->LINEAR->RELU->LINEAR->SOFTMAX.
    
    Arguments:
    X_train --обучающий набор данных, размером (входной размер объекта = 12288, кол-во обучающих примеров = 1080)
    Y_train -- обучающий набор меток, размером (выходной вектор = 6, кол-во примеров = 1080)
    X_test -- тестовый набор данных, размером (входной размер объекта = 12288, кол-во примеров = 120)
    Y_test -- тестовый набор меток, размером (выходной вектор = 6, кол-во примеров = 120)
    learning_rate -- скорость обучения
    num_epochs -- кол-во эпох (итерация) обучения
    minibatch_size -- размер мини-пакета
    print_cost -- True - печать значения функции потерь каждые 10 эпох
    Returns:
    parameters -- обученные параметры модели, которые можно использовать для прогнозирования
    """
    
    costs = []                                     
    train_acc = []
    test_acc = []
    
    # Инициализация параметров
    parameters = initialize_parameters()

    W1 = parameters['W1']
    b1 = parameters['b1']
    W2 = parameters['W2']
    b2 = parameters['b2']
    W3 = parameters['W3']
    b3 = parameters['b3']

    optimizer = tf.keras.optimizers.Adam(learning_rate)
    
    # CategoricalAccuracy метрика точности для многоклассовой классификации
    test_accuracy = tf.keras.metrics.CategoricalAccuracy()
    train_accuracy = tf.keras.metrics.CategoricalAccuracy()
    
    dataset = tf.data.Dataset.zip((X_train, Y_train))
    test_dataset = tf.data.Dataset.zip((X_test, Y_test))
    
    # Вычисления количество элеметнов в выборке
    m = dataset.cardinality().numpy()
    
    minibatches = dataset.batch(minibatch_size).prefetch(8)
    test_minibatches = test_dataset.batch(minibatch_size).prefetch(8)
    
    # Процесс обучения
    for epoch in range(num_epochs):

        epoch_cost = 0.
        
        # Сбрасываем значения объекта, чтобы начать измерение с 0 для каждой эпохи
        train_accuracy.reset_states()
        
        for (minibatch_X, minibatch_Y) in minibatches:
            
            with tf.GradientTape() as tape:
                # 1. predict
                Z3 = forward_propagation(tf.transpose(minibatch_X), parameters)

                # 2. loss
                minibatch_cost = compute_cost(Z3, tf.transpose(minibatch_Y))

            # Накапливаем точность всех партий
            train_accuracy.update_state(tf.transpose(Z3), minibatch_Y)
            
            trainable_variables = [W1, b1, W2, b2, W3, b3]
            grads = tape.gradient(minibatch_cost, trainable_variables)
            optimizer.apply_gradients(zip(grads, trainable_variables))
            epoch_cost += minibatch_cost
        
        # Делим стоимость эпохи на количество образцов
        epoch_cost /= m

        # Выводим стоимость каждые 10 эпох
        if print_cost == True and epoch % 10 == 0:
            print ("Cost after epoch %i: %f" % (epoch, epoch_cost))
            print("Train accuracy:", train_accuracy.result())
            
            # We evaluate the test set every 10 epochs to avoid computational overhead
            for (minibatch_X, minibatch_Y) in test_minibatches:
                Z3 = forward_propagation(tf.transpose(minibatch_X), parameters)
                test_accuracy.update_state(tf.transpose(Z3), minibatch_Y)
            print("Test_accuracy:", test_accuracy.result())

            costs.append(epoch_cost)
            train_acc.append(train_accuracy.result())
            test_acc.append(test_accuracy.result())
            test_accuracy.reset_states()


    return parameters, costs, train_acc, test_acc

In [ ]:
parameters, costs, train_acc, test_acc = model(new_train, new_y_train, new_test, new_y_test, num_epochs=100)

**Ожидаемый результат**: 

```
Cost after epoch 0: 0.057612
Train accuracy: tf.Tensor(0.17314816, shape=(), dtype=float32)
Test_accuracy: tf.Tensor(0.24166666, shape=(), dtype=float32)
Cost after epoch 10: 0.049332
Train accuracy: tf.Tensor(0.35833332, shape=(), dtype=float32)
Test_accuracy: tf.Tensor(0.3, shape=(), dtype=float32)
...
```
Числа, которые вы получаете, могут быть разными, просто убедитесь, что ваши потери уменьшаются, а ваша точность повышается.

In [ ]:
# График изменения значений функии потерь для каждой итерации
plt.plot(np.squeeze(costs))
plt.ylabel('Значение функции потерб')
plt.xlabel('Количество итераций')
plt.title("Скорость обучения =" + str(0.0001))
plt.grid(True)
plt.show()

In [ ]:
# График точности для обучающей выборки
plt.plot(np.squeeze(train_acc))
# График точности для тестовой выборки
plt.plot(np.squeeze(test_acc))
plt.legend(["Обучающая выборка", "Тестовая выборка"])
plt.title("Скорость обучения =" + str(0.0001))
plt.ylabel("Точность (accuracy)")
plt.xlabel("Количество итераций")
plt.grid(True)
plt.show()

Вот краткий обзор всего, чего вы только что достигли:

- Использовали `tf.Variable` для изменения ваших переменных
- Обучили нейронную сеть с помощью TensorFlow

Используемый материал:
- Курс Deep Learning; https://www.coursera.org/specializations/deep-learning
- Курс "Introduction to Gradients and Automatic Differentiation" https://www.tensorflow.org/guide/autodiff 
- Документация по GradientTape: https://www.tensorflow.org/api_docs/python/tf/GradientTape